In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm
import xml.etree.ElementTree as ET
#import patoolib
import zipfile
import re

ROOT_DIR = Path.cwd().parent.parent.parent

lkp_folder = os.path.join(ROOT_DIR, 'external_data','lookups','EHR','primary_care')

#from pomegranate.db.ukbdb import UKBDatabase
#Database = UKBDatabase()

# module I made
import dmd_xml_to_df

# Build mapping and lookup files

## Build consolidated files of all dm+d releases


In [ ]:
# on pCloud
dmd_archive_folder = os.path.join('X:/','XXXX','dm+d')
dmd_yr_folder_list = [x for x in os.listdir(dmd_archive_folder) if ('.rar' in x) or ('.zip' in x)]

# location for storing extracted files
out_dir = os.path.join('X:/','XXXX','XXXX','Downloads')
extract_folder = os.path.join(out_dir,'dmd_archive')

# dictionary to link files to the right functions
module_dict = {
    'lookup':dmd_xml_to_df.Lookup,
    'ingredient':dmd_xml_to_df.Ingredient,
    'vtm':dmd_xml_to_df.VTM,
    'vmp':dmd_xml_to_df.VMP,
    'vmpp':dmd_xml_to_df.VMPP,
    'amp':dmd_xml_to_df.AMP,
    'ampp':dmd_xml_to_df.AMPP
}

In [ ]:
extract_folder = os.path.join(out_dir,'dmd_archive')
for dmd_yr_folder in tqdm(dmd_yr_folder_list, desc='year'):
    year = dmd_yr_folder.split('_')[0]

    # skip the ones already done
    #if year in ['2014','2019','2021','2022','2023','2024']:
    #    continue

    # temporary folder to store extracted files
    os.makedirs(extract_folder, exist_ok=True)

    # extract
    patoolib.extract_archive(os.path.join(dmd_archive_folder,dmd_yr_folder), outdir=extract_folder)
    
    # makes a subfolder with the year
    release_dir = os.path.join(extract_folder,year)
    
    # process each release subfolder
    for release_folder in tqdm(os.listdir(release_dir), desc='release', leave=False):
        if os.path.isdir(os.path.join(release_dir,release_folder)):
            try:
                # only looking at main DMD releases rn
                dmd_folder = os.path.join(release_dir,release_folder,'NHSBSA_DMD')
                # get the release zip
                zip_file  = [x for x in os.listdir(dmd_folder) if re.match('nhsbsa_dmd_.*\.zip',x)]
                if len(zip_file) > 1:
                    print('multiple zip files found in the',year,'folder:', zip_file)
                    continue
                else:
                    zip_file = zip_file[0]
                    # extract
                    with zipfile.ZipFile(os.path.join(dmd_folder,zip_file), 'r') as zip_ref:
                        zip_ref.extractall(dmd_folder)
                    
                    # convert each xml to csv based on the module I made
                    for xml_file in [x for x in os.listdir(dmd_folder) 
                                    if ('.xml' in x) and ('trud_' not in x)]:
                        # xml files have this format: f_{type}2_{date}.xml
                        file_type = xml_file.split('_')[1][:-1]
                        if file_type in module_dict.keys():
                            dfs = module_dict[file_type](os.path.join(dmd_folder,xml_file)).output
                        else:
                            print('Non-standard file type in',dmd_folder,':',xml_file)
                            continue
                        # export each
                        for df_name in dfs.keys():
                            dfs[df_name].to_csv(os.path.join(dmd_folder,f'{df_name}.csv'), index=False)
            except FileNotFoundError:
                print('Issue with files in',dmd_folder)
                continue
        else:
            print('Non-folder file in', release_dir,':',release_folder)
            continue

It looks like they changed format in the 2021 folders after 3.4.0 - Switched from folders per release to just having the release zips in the main folder

Moving the zip files into folders to match the other years

In [ ]:
# put the zip files into their own release folders to match the old format
extract_folder = os.path.join(out_dir,'dmd_archive')
for dmd_yr in tqdm(['2021','2022','2023','2024'], desc='year'):
    dmd_yr_folder = os.path.join(extract_folder,dmd_yr)

    # pull out the zip files that need to be extracted and organized
    # move both the dmd and dmdbonus, but only process the dmd files
    dmd_zips  = [x for x in os.listdir(dmd_yr_folder) if re.match('nhsbsa_dmd_.*\.zip$',x)]

    for dmd_zip in dmd_zips:
        bonus_zip = dmd_zip.replace('_dmd_','_dmdbonus_')
        release = dmd_zip.split('_')[2]
        # first make the release folder
        os.makedirs(os.path.join(dmd_yr_folder,release), exist_ok=True)
        # then the dmd folder
        os.makedirs(os.path.join(dmd_yr_folder,release,'NHSBSA_DMD'), exist_ok=True)
        # move the zip in
        os.rename(os.path.join(dmd_yr_folder,dmd_zip), os.path.join(dmd_yr_folder,release,'NHSBSA_DMD',dmd_zip))
        # then the bonus folder
        os.makedirs(os.path.join(dmd_yr_folder,release,'NHSBSA_DMDBONUS'), exist_ok=True)
        # move the zip in
        os.rename(os.path.join(dmd_yr_folder,bonus_zip), os.path.join(dmd_yr_folder,release,'NHSBSA_DMDBONUS',bonus_zip))

Rerun now that the folders are moved

In [ ]:
extract_folder = os.path.join(out_dir,'dmd_archive')
for dmd_yr in tqdm(['2021','2022','2023','2024'], desc='year'):
    dmd_yr_folder = os.path.join(extract_folder,dmd_yr)
    
    # process each release subfolder
    for release_folder in tqdm(os.listdir(dmd_yr_folder), desc='release', leave=False):
        # skip those I've already processed
        if (dmd_yr=='2021') & (release_folder <= '3.4.0'):
            continue
        try:
            # only looking at main DMD releases rn
            dmd_folder = os.path.join(dmd_yr_folder,release_folder,'NHSBSA_DMD')
            # get the release zip
            zip_file  = [x for x in os.listdir(dmd_folder) if re.match('nhsbsa_dmd_.*\.zip',x)]
            if len(zip_file) > 1:
                print('multiple zip files found in the',year,'folder:', zip_file)
                continue
            else:
                zip_file = zip_file[0]
                # extract
                with zipfile.ZipFile(os.path.join(dmd_folder,zip_file), 'r') as zip_ref:
                    zip_ref.extractall(dmd_folder)
                
                # convert each xml to csv based on the module I made
                for xml_file in [x for x in os.listdir(dmd_folder) 
                                if ('.xml' in x) and ('trud_' not in x)]:
                    # xml files have this format: f_{type}2_{date}.xml
                    file_type = xml_file.split('_')[1][:-1]
                    if file_type in module_dict.keys():
                        dfs = module_dict[file_type](os.path.join(dmd_folder,xml_file)).output
                    else:
                        print('Non-standard file type in',dmd_folder,':',xml_file)
                        continue
                    # export each
                    for df_name in dfs.keys():
                        dfs[df_name].to_csv(os.path.join(dmd_folder,f'{df_name}.csv'), index=False)
        except FileNotFoundError:
            print('Issue with files in',dmd_folder)
            continue

Patch those with issues

In [ ]:
# simplified procedure since I know the two paths
zip_to_fix = {
    # this also had the 2018 12.2.0 release by accident - deleted
    os.path.join(extract_folder,'2019','12.2.0','NHSBSA_DMD'):'nhsbsa_dmd_12.2.0_20191216000001.zip',
    # just a weird duplication with an .asc ending - fixed the regex matching to exclude this in future
    os.path.join(extract_folder,'2016','3.1.0','NHSBSA_DMD'):'nhsbsa_dmd_3.1.0_20160314000001.zip'
}

for dmd_folder in zip_to_fix.keys():
    zip_file = zip_to_fix[dmd_folder]
    with zipfile.ZipFile(os.path.join(dmd_folder,zip_file), 'r') as zip_ref:
        zip_ref.extractall(dmd_folder)
    for xml_file in [x for x in os.listdir(dmd_folder) 
                    if ('.xml' in x) and ('trud_' not in x)]:
        file_type = xml_file.split('_')[1][:-1]
        if file_type in module_dict.keys():
            dfs = module_dict[file_type](os.path.join(dmd_folder,xml_file)).output
        else:
            print('Non-standard file type in',dmd_folder,':',xml_file)
            continue
        for df_name in dfs.keys():
            dfs[df_name].to_csv(os.path.join(dmd_folder,f'{df_name}.csv'), index=False)

### Combine release files

Concatenate each type of file and include the year and release. Then deduplicate and retain the first and last release mention of each entry

In [ ]:
def find_files_with_name(directory, target_name):
    path = Path(directory)
    return list(path.rglob(target_name))

file_types = [x for x in os.listdir('X:/XXXX/XXXXX/Downloads/dmd_archive/2014/1.0.0/NHSBSA_DMD') if '.csv' in x]
export_folder = os.path.join(dmd_archive_folder,'consolidated_2014_1.0.0_to_2024_7.2.0')

In [ ]:
file_types_already_run = [x.replace('_consolidated.csv','.csv') for x in os.listdir(export_folder) if '_consolidated.csv' in x]
file_types_to_run = list(set(file_types) - set(file_types_already_run))

for file_type in tqdm(file_types_to_run):
  outfile = os.path.join(export_folder,f'{file_type[:-4]}_consolidated.csv')
  if os.path.exists(outfile):
    continue
  else:
    try:
      # concat all files together
      matched_files = find_files_with_name(extract_folder, file_type)
      df = pd.DataFrame()
      for matched_file in matched_files:
        len_i = len(df)
        year = matched_file.parents[2].name
        release = matched_file.parents[1].name
        try:
          temp = pd.read_csv(matched_file, dtype='object')
        except pd.errors.EmptyDataError:
          #print(matched_file, 'is empty - skipping')
          continue
        if temp.empty:
            print(f"Skipping empty file: {matched_file}")
            continue
        temp['release'] = f'{year}_{release}'
        len_t = len(temp)
        df = pd.concat([df,temp], ignore_index=True)
        assert len(df) == len_t + len_i
    except Exception as e:
      print('Concat issue with',file_type, e)

    if df.empty:
      print('The concatenated file for', file_type, 'is empty')
      continue

    # deduplicate and store earliest and latest release's mention
    df = (df
            .groupby(list(df.columns[:-1]), as_index=False, dropna=False)
            .agg(
              earliest_release=('release','min'),
              latest_release=('release','max')
              )
    )

    # export this to the pCloud folder
    df.to_csv(outfile, index=False)
    print('Exported',f'{file_type[:-4]}_consolidated.csv')

### Check out the empty files?

In [ ]:
empty_files = [
    'lookup_DfIndInfoType.csv',
    'lookup_DtPayCatInfoType.csv',
    'lookup_UoMHistoryInfoType.csv',
    'lookup_DtInfoType.csv',
    'lookup_ReimbStatInfoType.csv',
    'lookup_DiscIndInfoType.csv',
    'lookup_FormHistoryInfoType.csv',
    'lookup_LicAuthInfoType.csv',
    'lookup_RouteHistoryInfoType.csv',
    'lookup_VirProdNoAvailInfoType.csv',
    'lookup_LicAuthChgRsnInfoType.csv',
    'lookup_DrugStatusInfoType.csv'
]

## dm+d lookups

All consolidated files copied to the shared folder, so read them in from there

AIM 1: Make these files more useful and easy to understand by mapping their column names to something more sensible, and exporting them into the same folder

AIM 2: identify the 'reference' codes - those active in the most recent dm+d release

AIM 3: Make files needed for prescribing unification

In [ ]:
src_folder = os.path.join('X:/','XXXXXX','TRUD_maps','dmd_consolidated_2014_1.0.0_to_2024_7.2.0')
# place to put the translated files
trans_folder = os.path.join('X:/','XXXXX','TRUD_maps','dmd_consolidated_clarified_2014_1.0.0_to_2024_7.2.0')

### VTM

Simple, only a single file

In [ ]:
vtm_src = pd.read_csv(os.path.join(src_folder,'vtm_consolidated.csv'), dtype=object)

In [ ]:
vtm_col_map = {
    'VTMID':'vtm_code',
    'NM':'vtm_desc',
    'VTMIDPREV':'previous_vtm_code',
    'VTMIDDT':'valid_date',
    'INVALID':'invalid_flag'
}

In [ ]:
vtm_lkp = vtm_src.rename(columns=vtm_col_map)

EXPORT

In [ ]:
# export translated
vtm_lkp.to_csv(os.path.join(trans_folder,'vtm_lookup.csv'), index=False)
# export to repo
vtm_lkp.to_csv(os.path.join(lkp_folder,'dmd_vtm_lkp.csv.gz'), compression='gzip', index=False)

### VMP

There are 6 VMP datasets. Map the corresponding lookups where possible for ease

- ControlInfoType
    - "Information relating to Virtual Medicinal Products where these are drugs and in particular where the drug is controlled under the Misuse of Drugs Act."
    - Not needed for DB lookup
- DrugFormType ***TODO***
    - "Information relating the Virtual Medicinal Product to its categorical form, which is defined as the form in which it is normally **dispensed**."
    - Not needed for DB lookup
- DrugRouteType ***TODO***
    - "the place in or on the body where the product **is introduced** in order to achieve the desired therapeutic effect."
    - Not needed for DB lookup
- OntDrugFormType ***TODO***
    - "Information relating the VMP to its combined form and route of administration. The form is the form of the product **at administration** to the patient e.g. a soluble tablet will be a solution."
    - Not needed for DB lookup
- VmpType
    - The main VMP lookup
    - To put in repo
- VpiType
    - "Information about an ingredient substance that is contained within a Virtual Medicinal Product.
This will include substances which are deemed to be ‘active’ ingredients.  Excipient information is associated with Actual Medicinal Products.
An example of a multiple virtual product ingredient substances entry:  co-amoxiclav 250mg/125mg would have two entries, one for amoxicillin 250mg, the other for clavulanic acid 125mg."
    - To put in the repo

#### Controlled Drugs

In [ ]:
vmp_control = pd.read_csv(os.path.join(src_folder,'vmp_ControlInfoType_consolidated.csv'), dtype=object)

In [ ]:
vmp_control_map = {
    'VPID':'vmp_code',
    'CATCD':'control_drug_cat',
    'CATDT':'valid_date',
    'CAT_PREVCD':'previous_control_drug_cat'
}

In [ ]:
vmp_control = vmp_control.rename(columns=vmp_control_map)

Merge in the control information

In [ ]:
control_key = pd.read_csv(os.path.join(src_folder,'lookup_ControlDrugCatInfoType_consolidated.csv'), dtype=object)

In [ ]:
vmp_control = vmp_control.merge(
    control_key[['CD','DESC']],
    how='left',
    left_on='control_drug_cat',
    right_on='CD'
)
# replace the category code with the description
vmp_control['control_drug_cat'] = vmp_control['DESC']
vmp_control.drop(columns=['CD','DESC'], inplace=True)

In [ ]:
# export translated
vmp_control.to_csv(os.path.join(trans_folder,'vmp_controlled_drug_categories.csv'), index=False)

#### VMP lookup

Need to pull info from lookups:
- basis of name
- name change reason
- combination product
- non-availability
- unit of measure
- dose form indicator
- prescribing status

In [ ]:
vmp_lkp = pd.read_csv(os.path.join(src_folder,'vmp_VmpType_consolidated.csv'), dtype=object)

In [ ]:
# first just translate columns
vmp_lkp_map = {
    'VPID':'vmp_code',
    'VPIDDT':'valid_date',
    'VPIDPREV':'previous_vmp_code',
    'VTMID':'vtm_code',
    'INVALID':'invalid_flag',
    'NM':'vmp_desc', # keep the prefix to retain ID when merging with other dm+d files
    'ABBREVNM':'abbrev_desc', # abbreviated description
    'BASISCD':'preferred_desc_basis', # CODE TO MAP TO LOOKUP
    'NMDT':'desc_valid_date', # date name applied
    'NMPREV':'previous_desc', 
    'BASIS_PREVCD':'previous_desc_basis', # CODE TO MAP TO LKP
    'NMCHANGECD':'desc_change_reason', # CODE TO MAP TO LKP
    'COMBPRODCD':'combination_product', # combination product indicator
    'PRES_STATCD':'prescribing_status', # CODE TO MAP TO LKP
    'SUG_F':'sugar_free',
    'GLU_F':'gluten_free',
    'PRES_F':'preservative_free',
    'CFC_F':'cfc_free',
    'NON_AVAILCD':'amp_non_avail', # CODE TO MAP TO LKP
    'NON_AVAILDT':'amp_non_avail_date', # non availability status date
    'DF_INDCD':'dose_form', # CODE TO MAP TO LKP
    'UDFS':'unit_dose_form_size',
    'UDFS_UOMCD':'unit_dose_form_units', # CODE TO MAP TO LKP
    'UNIT_DOSE_UOMCD':'unit_dose_meas_unit' # unit dose unit of measure # CODE TO MAP TO LKP
}

vmp_lkp = vmp_lkp.rename(columns=vmp_lkp_map)

##### Map the coded terms to their meanings

Basis of name

In [ ]:
basis_name_key = pd.read_csv(os.path.join(src_folder,'lookup_BasisOfNameInfoType_consolidated.csv'),dtype=object)

In [ ]:
# current basis of name
vmp_lkp = vmp_lkp.merge(
    basis_name_key[['CD','DESC']],
    how='left',
    left_on='preferred_desc_basis',
    right_on='CD'
)
# replace the category code with the description
vmp_lkp['preferred_desc_basis'] = vmp_lkp['DESC']
vmp_lkp.drop(columns=['CD','DESC'], inplace=True)

In [ ]:
# previous basis of name
vmp_lkp = vmp_lkp.merge(
    basis_name_key[['CD','DESC']],
    how='left',
    left_on='previous_desc_basis',
    right_on='CD'
)
# replace the category code with the description
vmp_lkp['previous_desc_basis'] = vmp_lkp['DESC']
vmp_lkp.drop(columns=['CD','DESC'], inplace=True)

Name change reason

In [ ]:
name_change_key = pd.read_csv(os.path.join(src_folder,'lookup_NamechangeReasonInfoType_consolidated.csv'),dtype=object)

In [ ]:
vmp_lkp = vmp_lkp.merge(
    name_change_key[['CD','DESC']],
    how='left',
    left_on='desc_change_reason',
    right_on='CD'
)
# replace the category code with the description
vmp_lkp['desc_change_reason'] = vmp_lkp['DESC']
vmp_lkp.drop(columns=['CD','DESC'], inplace=True)

Combination product indicator

In [ ]:
comb_prod_key = pd.read_csv(os.path.join(src_folder,'lookup_CombProdIndInfoType_consolidated.csv'),dtype=object)

In [ ]:
vmp_lkp = vmp_lkp.merge(
    comb_prod_key[['CD','DESC']],
    how='left',
    left_on='combination_product',
    right_on='CD'
)
# replace the category code with the description
vmp_lkp['combination_product'] = vmp_lkp['DESC']
vmp_lkp.drop(columns=['CD','DESC'], inplace=True)

Prescribing status

In [ ]:
presc_status_key = pd.read_csv(os.path.join(src_folder,'lookup_VirProdPresStatInfoType_consolidated.csv'),dtype=object)

In [ ]:
vmp_lkp = vmp_lkp.merge(
    presc_status_key[['CD','DESC']],
    how='left',
    left_on='prescribing_status',
    right_on='CD'
)
# replace the category code with the description
vmp_lkp['prescribing_status'] = vmp_lkp['DESC']
vmp_lkp.drop(columns=['CD','DESC'], inplace=True)

Non-availability


> A flag indicating whether there are currently no available Actual Medicinal Products which correspond to this VMP.

VALUES:  
- 0 = actual products are now available (though not necessarily prescribable in primary care) but have been unavailable in the past.
- 1 = actual products no longer available
- Null = AMPs are available now


In [ ]:
non_avail_map = {
    np.nan: 'AMPs are available now',
    '0000':'AMPs are available now but were previously unavailable',
    '0001': 'AMPs no longer available'
}
vmp_lkp['amp_non_avail'] = vmp_lkp['amp_non_avail'].map(non_avail_map)

Dose form indicator

> This attribute identifies if the VMP has a unit dose form (a discrete unit dose is applicable e.g. tablet, ampoule), if it may be regarded as a continuous substance (that is a consistent physically measurable unit or sub-unit cannot be defined e.g. cream, eye drops) or if it belongs to a category of product for which unit dose form is not appropriate (catheters, colostomy bags, ACBS foods)

VALUES:
- 1 = Discrete
- 2 = Continuous
- 3 = Not applicable


In [ ]:
dose_form_map = {
    '1':'Discrete',
    '2':'Continuous',
    '3':'NA'
}
vmp_lkp['dose_form'] = vmp_lkp['dose_form'].map(dose_form_map)

Unit of Measure



In [ ]:
oum_key = pd.read_csv(os.path.join(src_folder,'lookup_UoMHistoryInfoType.csv'),dtype=object)

In [ ]:
vmp_lkp = vmp_lkp.merge(
    oum_key[['CD','DESC']],
    how='left',
    left_on='unit_dose_form_units',
    right_on='CD'
)
# replace the category code with the description
vmp_lkp['unit_dose_form_units'] = vmp_lkp['DESC']
vmp_lkp.drop(columns=['CD','DESC'], inplace=True)

In [ ]:
vmp_lkp = vmp_lkp.merge(
    oum_key[['CD','DESC']],
    how='left',
    left_on='unit_dose_meas_unit',
    right_on='CD'
)
# replace the category code with the description
vmp_lkp['unit_dose_meas_unit'] = vmp_lkp['DESC']
vmp_lkp.drop(columns=['CD','DESC'], inplace=True)

EXPORT

In [ ]:
# export translated
vmp_lkp.to_csv(os.path.join(trans_folder,'vmp_lookup.csv'), index=False)

In [ ]:
# export to repo
vmp_lkp.to_csv(os.path.join(lkp_folder,'dmd_vmp_info_lkp.csv.gz'), compression='gzip', index=False)

#### Ingredient

Requires lookups:
- ingredient
- unit of measure
- basis of strength

In [ ]:
vmp_ing = pd.read_csv(os.path.join(src_folder,'vmp_VpiType_consolidated.csv'), dtype=object)

In [ ]:
vmp_ing_map = {
    'VPID':'vmp_code',
    'ISID':'ingredient',
    'BASIS_STRNTCD':'strength_basis',
    'BS_SUBID':'strength_substance_basis',
    'STRNT_NMRTR_VAL':'strength_numerator',
    'STRNT_NMRTR_UOMCD':'strength_numerator_units',
    'STRNT_DNMTR_VAL':'strength_denominator',
    'STRNT_DNMTR_UOMCD':'strength_denominator_units'
}

In [ ]:
vmp_ing = vmp_ing.rename(columns=vmp_ing_map)

Merge ingredient name in

In [ ]:
ing_lkp = pd.read_csv(os.path.join(src_folder,'ingredient_consolidated.csv'), dtype=object)

In [ ]:
vmp_ing = vmp_ing.merge(
    ing_lkp[['ISID','NM']],
    how='left',
    left_on='ingredient',
    right_on='ISID'
)
# replace the category code with the description
vmp_ing['ingredient'] = vmp_ing['NM']
vmp_ing.drop(columns=['ISID','NM'], inplace=True)

In [ ]:
# basis of strength substance
vmp_ing = vmp_ing.merge(
    ing_lkp[['ISID','NM']],
    how='left',
    left_on='strength_substance_basis',
    right_on='ISID'
)
# replace the category code with the description
vmp_ing['strength_substance_basis'] = vmp_ing['NM']
vmp_ing.drop(columns=['ISID','NM'], inplace=True)

Merge unit of measure

In [ ]:
vmp_ing = vmp_ing.merge(
    oum_key[['CD','DESC']],
    how='left',
    left_on='strength_numerator_units',
    right_on='CD'
)
# replace the category code with the description
vmp_ing['strength_numerator_units'] = vmp_ing['DESC']
vmp_ing.drop(columns=['CD','DESC'], inplace=True)

In [ ]:
vmp_ing = vmp_ing.merge(
    oum_key[['CD','DESC']],
    how='left',
    left_on='strength_denominator_units',
    right_on='CD'
)
# replace the category code with the description
vmp_ing['strength_denominator_units'] = vmp_ing['DESC']
vmp_ing.drop(columns=['CD','DESC'], inplace=True)

Strength basis

0001 = based on ingredient substance

0002 = based on base substance

In [ ]:
strength_basis_map = {
    '0001': 'based on ingredient substance',
    '0002': 'based on base substance'
}
vmp_ing['strength_basis'] = vmp_ing['strength_basis'].map(strength_basis_map)

EXPORT

In [ ]:
# export translated
vmp_ing.to_csv(os.path.join(trans_folder,'vmp_ingredient.csv'), index=False)

In [ ]:
# export to repo
vmp_ing.to_csv(os.path.join(lkp_folder,'dmd_vmp_ingredient_lkp.csv.gz'), compression='gzip', index=False)

### VMPP


#### VMPP lookup

Needs lookups:
- unit of measure
- combination pack

In [ ]:
vmpp_lkp = pd.read_csv(os.path.join(src_folder,'vmpp_VmppType_consolidated.csv'), dtype=object)

In [ ]:
vmpp_lkp_map = {
    'VPPID':'vmpp_code', 
    'INVALID':'invalid_flag', 
    'NM':'vmpp_desc', 
    'ABBREVNM':'abbrev_desc', 
    'VPID':'vmp_code',
    'QTYVAL':'quantity', 
    'QTY_UOMCD':'quantity_units', # NEEDS LKP MAP
    'COMBPACKCD':'combination_pack' # NEEDS LKP MAP
}

vmpp_lkp = vmpp_lkp.rename(columns=vmpp_lkp_map)

Quantity units

In [ ]:
vmpp_lkp = vmpp_lkp.merge(
    oum_key[['CD','DESC']],
    how='left',
    left_on='quantity_units',
    right_on='CD'
)
# replace the category code with the description
vmpp_lkp['quantity_units'] = vmpp_lkp['DESC']
vmpp_lkp.drop(columns=['CD','DESC'], inplace=True)

Combination pack

In [ ]:
comb_pack_key = pd.read_csv(os.path.join(src_folder,'lookup_CombPackIndInfoType_consolidated.csv'), dtype=object)

In [ ]:
vmpp_lkp = vmpp_lkp.merge(
    comb_pack_key[['CD','DESC']],
    how='left',
    left_on='combination_pack',
    right_on='CD'
)
# replace the category code with the description
vmpp_lkp['combination_pack'] = vmpp_lkp['DESC']
vmpp_lkp.drop(columns=['CD','DESC'], inplace=True)

EXPORT

In [ ]:
# export translated
vmpp_lkp.to_csv(os.path.join(trans_folder,'vmpp_lookup.csv'), index=False)
# export to repo
vmpp_lkp.to_csv(os.path.join(lkp_folder,'dmd_vmpp_info_lkp.csv.gz'), compression='gzip', index=False)

### AMP

There are 4 datasets

- AMP general lookup
    - To add to repo
- Ingredient lookup ***TODO***
    - Not really needed since we can link from AMP to VMP and get them from the VMP ingredient lookup
    - NOT for repo
- Product info ***TODO***
- Licensed Route ***TODO***

#### AMP lookup

Needs lookups:
- supplier
- licensing authority
- licensing authority change reason
- combination product
- flavor
- availability restrictions

In [ ]:
amp_lkp = pd.read_csv(os.path.join(src_folder,'amp_AmpType_consolidated.csv'), dtype=object)

In [ ]:
amp_lkp_map = {
    'APID':'amp_code', 
    'INVALID':'invalid_flag', 
    'VPID':'vmp_code', 
    'NM':'amp_desc', 
    'ABBREVNM':'abbrev_desc', 
    'DESC':'long_desc', 
    'NMDT':'desc_valid_date', 
    'NM_PREV':'previous_desc',
    'SUPPCD':'supplier', # NEEDS LKP MAP 
    'LIC_AUTHCD':'license_authority', # NEEDS LKP MAP 
    'LIC_AUTH_PREVCD':'previous_license_authority', # NEEDS LKP MAP 
    'LIC_AUTHCHANGECD':'license_authority_change_reason', # NEEDS LKP MAP 
    'LIC_AUTHCHANGEDT':'license_authority_change_date', 
    'COMBPRODCD':'combination_product',  # NEEDS LKP MAP 
    'FLAVOURCD':'flavour',  # NEEDS LKP MAP 
    'CSM':'ema_monitoring', # European Medicines Agency monitoring - replaced CSM in 2013
    'PARALLEL_IMPORT':'imported_from_eu',
    'AVAIL_RESTRICTCD':'avail_restrictions' # NEEDS LKP MAP 
}

amp_lkp = amp_lkp.rename(columns=amp_lkp_map)

Suppliers

In [ ]:
supplier_key = pd.read_csv(os.path.join(src_folder,'lookup_SupplierInfoType_consolidated.csv'), dtype=object)

In [ ]:
# the names change over time so use the most recent name
supplier_key = (supplier_key
                .sort_values('earliest_release', ascending=False)
                .drop_duplicates(subset='CD',keep='first'))

In [ ]:
amp_lkp = amp_lkp.merge(
    supplier_key[['CD','DESC']],
    how='left',
    left_on='supplier',
    right_on='CD'
)
# replace the category code with the description
amp_lkp['supplier'] = amp_lkp['DESC']
amp_lkp.drop(columns=['CD','DESC'], inplace=True)

Licensing authority

Using info from a recent release file

In [ ]:
lic_auth_key = pd.read_csv(os.path.join(src_folder,'lookup_LicAuthIndInfoType.csv'), dtype=object)

In [ ]:
amp_lkp = amp_lkp.merge(
    lic_auth_key[['CD','DESC']],
    how='left',
    left_on='license_authority',
    right_on='CD'
)
# replace the category code with the description
amp_lkp['license_authority'] = amp_lkp['DESC']
amp_lkp.drop(columns=['CD','DESC'], inplace=True)

In [ ]:
amp_lkp = amp_lkp.merge(
    lic_auth_key[['CD','DESC']],
    how='left',
    left_on='previous_license_authority',
    right_on='CD'
)
# replace the category code with the description
amp_lkp['previous_license_authority'] = amp_lkp['DESC']
amp_lkp.drop(columns=['CD','DESC'], inplace=True)

Licensing authority change reason

Using info from a recent release file

In [ ]:
lic_auth_chg_key = pd.read_csv(os.path.join(src_folder,'lookup_LicAuthChgRsnInfoType.csv'), dtype=object)

In [ ]:
amp_lkp = amp_lkp.merge(
    lic_auth_chg_key[['CD','DESC']],
    how='left',
    left_on='license_authority_change_reason',
    right_on='CD'
)
# replace the category code with the description
amp_lkp['license_authority_change_reason'] = amp_lkp['DESC']
amp_lkp.drop(columns=['CD','DESC'], inplace=True)

Combination product

In [ ]:
amp_lkp = amp_lkp.merge(
    comb_prod_key[['CD','DESC']],
    how='left',
    left_on='combination_product',
    right_on='CD'
)
# replace the category code with the description
amp_lkp['combination_product'] = amp_lkp['DESC']
amp_lkp.drop(columns=['CD','DESC'], inplace=True)

Flavour

In [ ]:
flavor_key = pd.read_csv(os.path.join(src_folder,'lookup_FlavourInfoType_consolidated.csv'), dtype=object)

In [ ]:
# some names changed over time so use the most recent name
flavor_key = (flavor_key
                .sort_values('earliest_release', ascending=False)
                .drop_duplicates(subset='CD',keep='first'))

In [ ]:
amp_lkp = amp_lkp.merge(
    flavor_key[['CD','DESC']],
    how='left',
    left_on='flavour',
    right_on='CD'
)
# replace the category code with the description
amp_lkp['flavour'] = amp_lkp['DESC']
amp_lkp.drop(columns=['CD','DESC'], inplace=True)

Availability restrictions

In [ ]:
avail_rest_key = pd.read_csv(os.path.join(src_folder,'lookup_AvailRestrictInfoType.csv'), dtype=object)

In [ ]:
amp_lkp = amp_lkp.merge(
    avail_rest_key[['CD','DESC']],
    how='left',
    left_on='avail_restrictions',
    right_on='CD'
)
# replace the category code with the description
amp_lkp['avail_restrictions'] = amp_lkp['DESC']
amp_lkp.drop(columns=['CD','DESC'], inplace=True)

EXPORT

In [ ]:
# export translated
amp_lkp.to_csv(os.path.join(trans_folder,'amp_lookup.csv'), index=False)
# export to repo
amp_lkp.to_csv(os.path.join(lkp_folder,'dmd_amp_info_lkp.csv.gz'), compression='gzip', index=False)

### AMPP

There are 6 datasets

JUST DOING THE DESCRIPTION FOR NOW

- AMP lookup
- Content ***TODO***
- Pack ***TODO***
- Prescribing ***TODO***
- Price ***TODO***
- Reimbursement ***TODO***

#### AMPP lookup

Needed lookups:
- combination pack
- legal category
- discontinued

In [ ]:
ampp_lkp = pd.read_csv(os.path.join(src_folder,'ampp_AmppType_consolidated.csv'), dtype=object)

In [ ]:
ampp_lkp_map = {
    'APPID':'ampp_code', 
    'INVALID':'invalid_flag', 
    'NM':'ampp_desc', 
    'ABBREVNM':'abbrev_desc', 
    'VPPID':'vmpp_code',
    'APID':'amp_code', 
    'COMBPACKCD':'combination_pack', # NEEDS LKP MAP
    'LEGAL_CATCD':'legal_cat', 
    'SUBP':'subpack', 
    'DISCCD':'discontinued', 
    'DISCDT':'discontinued_date'
}

ampp_lkp = ampp_lkp.rename(columns=ampp_lkp_map)

Combination pack

In [ ]:
ampp_lkp = ampp_lkp.merge(
    comb_pack_key[['CD','DESC']],
    how='left',
    left_on='combination_pack',
    right_on='CD'
)
# replace the category code with the description
ampp_lkp['combination_pack'] = ampp_lkp['DESC']
ampp_lkp.drop(columns=['CD','DESC'], inplace=True)

Legal category

In [ ]:
legal_key = pd.read_csv(os.path.join(src_folder,'lookup_LegalCatInfoType_consolidated.csv'), dtype=object)

In [ ]:
ampp_lkp = ampp_lkp.merge(
    legal_key[['CD','DESC']],
    how='left',
    left_on='legal_cat',
    right_on='CD'
)
# replace the category code with the description
ampp_lkp['legal_cat'] = ampp_lkp['DESC']
ampp_lkp.drop(columns=['CD','DESC'], inplace=True)

Discontinued

File copied from another release

In [ ]:
discontinued_key = pd.read_csv(os.path.join(src_folder,'lookup_DiscIndInfoType.csv'), dtype=object)

In [ ]:
ampp_lkp = ampp_lkp.merge(
    discontinued_key[['CD','DESC']],
    how='left',
    left_on='discontinued',
    right_on='CD'
)
# replace the category code with the description
ampp_lkp['discontinued'] = ampp_lkp['DESC']
ampp_lkp.drop(columns=['CD','DESC'], inplace=True)

EXPORT

In [ ]:
# export translated
ampp_lkp.to_csv(os.path.join(trans_folder,'ampp_lookup.csv'), index=False)
# export to repo
ampp_lkp.to_csv(os.path.join(lkp_folder,'dmd_ampp_info_lkp.csv.gz'), compression='gzip', index=False)

### MAIN DM+D KEY

All possible descriptions and types of each code, for identifying codes.

#### VTM

In [ ]:
vtm_desc_lkp = vtm_lkp[['vtm_code','vtm_desc','most_recent_by_desc']].dropna().rename(columns={'vtm_code':'dmd_code','vtm_desc':'desc'}).drop_duplicates()
vtm_desc_lkp['dmd_type'] = 'vtm'
vtm_desc_lkp['desc_type'] = 'preferred'

#### VMP

Include both the full description and the abbreviated description

In [ ]:
vmp_desc = vmp_lkp[['vmp_code','vmp_desc','most_recent_by_desc']].dropna().rename(columns={'vmp_code':'dmd_code','vmp_desc':'desc'}).drop_duplicates()
vmp_desc['desc_type'] = 'preferred'
vmp_abbrev_desc = vmp_lkp[['vmp_code','abbrev_desc','most_recent_by_desc']].dropna().rename(columns={'vmp_code':'dmd_code','abbrev_desc':'desc'}).drop_duplicates()
vmp_abbrev_desc['desc_type'] = 'abbreviated'
# combine
vmp_desc_lkp = pd.concat([vmp_desc,vmp_abbrev_desc])
vmp_desc_lkp['dmd_type'] = 'vmp'

#### VMPP

In [ ]:
vmpp_desc = vmpp_lkp[['vmpp_code','vmpp_desc','most_recent_by_desc']].dropna().rename(columns={'vmpp_code':'dmd_code','vmpp_desc':'desc'}).drop_duplicates()
vmpp_desc['desc_type'] = 'preferred'
vmpp_abbrev_desc = vmpp_lkp[['vmpp_code','abbrev_desc','most_recent_by_desc']].dropna().rename(columns={'vmpp_code':'dmd_code','abbrev_desc':'desc'}).drop_duplicates()
vmpp_abbrev_desc['desc_type'] = 'abbreviated'
# combine
vmpp_desc_lkp = pd.concat([vmpp_desc,vmpp_abbrev_desc])
vmpp_desc_lkp['dmd_type'] = 'vmpp'

#### AMP

This also has a long description

In [ ]:
amp_desc = amp_lkp[['amp_code','amp_desc','most_recent_by_desc']].dropna().rename(columns={'amp_code':'dmd_code','amp_desc':'desc'}).drop_duplicates()
amp_desc['desc_type'] = 'preferred'
amp_abbrev_desc = amp_lkp[['amp_code','abbrev_desc','most_recent_by_desc']].dropna().rename(columns={'amp_code':'dmd_code','abbrev_desc':'desc'}).drop_duplicates()
amp_abbrev_desc['desc_type'] = 'abbreviated'
amp_long_desc = amp_lkp[['amp_code','long_desc','most_recent_by_desc']].dropna().rename(columns={'amp_code':'dmd_code','long_desc':'desc'}).drop_duplicates()
amp_long_desc['desc_type'] = 'long'
# combine
amp_desc_lkp = pd.concat([amp_desc,amp_abbrev_desc,amp_long_desc])
amp_desc_lkp['dmd_type'] = 'amp'

#### AMPP

In [ ]:
ampp_desc = ampp_lkp[['ampp_code','ampp_desc','most_recent_by_desc']].dropna().rename(columns={'ampp_code':'dmd_code','ampp_desc':'desc'}).drop_duplicates()
ampp_desc['desc_type'] = 'preferred'
ampp_abbrev_desc = ampp_lkp[['ampp_code','abbrev_desc','most_recent_by_desc']].dropna().rename(columns={'ampp_code':'dmd_code','abbrev_desc':'desc'}).drop_duplicates()
ampp_abbrev_desc['desc_type'] = 'abbreviated'
# combine
ampp_desc_lkp = pd.concat([ampp_desc,ampp_abbrev_desc])
ampp_desc_lkp['dmd_type'] = 'ampp'

#### Merge

In [ ]:
dmd_desc_lkp = pd.concat([
    vtm_desc_lkp, vmp_desc_lkp, amp_desc_lkp, vmpp_desc_lkp, ampp_desc_lkp
], ignore_index=True)

# export
dmd_desc_lkp.to_csv(os.path.join(lkp_folder,'dmd_desc_type_lkp.csv.gz'), compression='gzip', index=False)

### Reference code key

A simple file to map all versions of the same code to the most current version. 

From NHSBSA: 
>Note: the dm+d codes that can change are VTM, VMP, Form, Route, Ingredient, Supplier and Unit of measure.
 

To be used for aligning the 'dmd_code' column of the new harmonized table, and as an input to a function that pulls all equivalent dm+d codes for a given code.

In [ ]:
vtm_lkp = pd.read_csv(os.path.join(trans_folder,'vtm_lookup.csv'), dtype=object)
vmp_lkp = pd.read_csv(os.path.join(trans_folder,'vmp_lookup.csv'), dtype=object)
amp_lkp = pd.read_csv(os.path.join(trans_folder,'amp_lookup.csv'), dtype=object)
ampp_lkp = pd.read_csv(os.path.join(trans_folder,'ampp_lookup.csv'), dtype=object)
vmpp_lkp = pd.read_csv(os.path.join(trans_folder,'vmpp_lookup.csv'), dtype=object)

Test - update codes

In [ ]:
# test
test_df = pd.DataFrame({
    'current_code':[4,3,2,1,'A','B','C'],
    'previous_code':[3,2,1,np.nan,np.nan,'C',np.nan]
})

# find the codes that need to be updated
test_df_nona = test_df[~test_df['previous_code'].isna()].copy()
test_map = dict(zip(test_df_nona["previous_code"], test_df_nona["current_code"]))

def find_test_reference(code):
    # if a previous code
    while code in test_map:
        code = test_map[code]
    # return the final resolved reference code
    return code

test_df['reference_code'] = test_df['current_code'].apply(find_test_reference)

display(test_df)

In [ ]:
# because the only codes that can update are vmp or vtm, combine
vtm_lkp_nona = vtm_lkp[~vtm_lkp['previous_vtm_code'].isna()].copy()
vmp_lkp_nona = vmp_lkp[~vmp_lkp['previous_vmp_code'].isna()].copy()

vtm_old_mapping = dict(zip(vtm_lkp_nona["previous_vtm_code"], vtm_lkp_nona["vtm_code"]))
vmp_old_mapping = dict(zip(vmp_lkp_nona["previous_vmp_code"], vmp_lkp_nona["vmp_code"]))

Apply for VTM and VMP

In [ ]:
def find_vtm_reference(code):
    # if a previous code
    while code in vtm_old_mapping:
        code_i = code
        code = vtm_old_mapping[code]
        try:
            code_j = vtm_old_mapping[code]
            if code_i == code_j:
                # this means that the code was replaced with an old version of a code
                # use the most recently valid
                code = vtm_lkp[vtm_lkp['vtm_code'].isin([code_i,code])].sort_values('valid_date', ascending=False)['vtm_code'].values[0]
                break
        except KeyError:
            continue
    # return the final resolved reference code
    return code

vtm_lkp.insert(1,'reference_vtm_code', vtm_lkp['vtm_code'].apply(find_vtm_reference))

*Example of codes that are self-referencing*

In [ ]:
vtm_lkp[vtm_lkp['vtm_code'] == '10047111000001104']

In [ ]:
vtm_lkp[vtm_lkp['vtm_code'] == '24891006']

In [ ]:
def find_vmp_reference(code):
    # if a previous code
    while code in vmp_old_mapping:
        code_i = code
        code = vmp_old_mapping[code]
        try:
            code_j = vmp_old_mapping[code]
            if code_i == code_j:
                print('Conflict',code_i, code)
                break
                # this means that the code was replaced with an old version of a code
                # use the most recently valid
                #code = vtm_lkp[vtm_lkp['vtm_code'].isin([code_i,code])].sort_values('valid_date', ascending=False)['vtm_code'].values[0]
        except KeyError:
            continue
    # return the final resolved reference code
    return code

vmp_lkp.insert(1,'reference_vmp_code', vmp_lkp['vmp_code'].apply(find_vmp_reference))

Add the reference code to the lookup (only relevant for VTM and VMP but useful as a source file)

In [ ]:
dmd_desc_lkp = pd.read_csv(os.path.join(lkp_folder,'dmd_desc_type_lkp.csv.gz'), compression='gzip', dtype=object)

In [ ]:
# reference info to add
ref_info = pd.concat([
    (vtm_lkp
    [['vtm_code','reference_vtm_code']]
    .rename(columns={'vtm_code':'dmd_code','reference_vtm_code':'reference_code'})),
    (vmp_lkp
    [['vmp_code','reference_vmp_code']]
    .rename(columns={'vmp_code':'dmd_code','reference_vmp_code':'reference_code'}))
])

# merge the reference cols in
dmd_desc_lkp = dmd_desc_lkp.merge(ref_info, how='left')

# just copy the codes if na (for AMP, AMPP, VMPP)
dmd_desc_lkp['reference_code'] = dmd_desc_lkp.apply(lambda row: row['dmd_code'] if pd.isna(row['reference_code']) else row['reference_code'], axis=1)

# drop duplicates
dmd_desc_lkp.drop_duplicates(inplace=True)

Add the reference codes to the lookups that refer to VTM and VMP

In [ ]:
amp_lkp = amp_lkp.merge(vmp_lkp[['vmp_code','reference_vmp_code']], how='left', on='vmp_code')
#reorganize columns
amp_lkp = amp_lkp[['amp_code', 'invalid_flag', 'vmp_code',
       'reference_vmp_code', 'amp_desc', 'abbrev_desc',
       'long_desc', 'desc_valid_date', 'previous_desc', 'supplier',
       'license_authority', 'previous_license_authority',
       'license_authority_change_reason', 'license_authority_change_date',
       'combination_product', 'flavour', 'ema_monitoring', 'imported_from_eu',
       'avail_restrictions', 'earliest_release', 'latest_release']]

In [ ]:
vmpp_lkp = vmpp_lkp.merge(vmp_lkp[['vmp_code','reference_vmp_code']], how='left', on='vmp_code')
vmpp_lkp = vmpp_lkp[[
    'vmpp_code', 'invalid_flag', 'vmpp_desc', 'abbrev_desc', 'vmp_code','reference_vmp_code',
    'quantity', 'quantity_units', 'combination_pack', 'earliest_release','latest_release'
    ]]

In [ ]:
vmp_lkp = vmp_lkp.merge(vtm_lkp[['vtm_code','reference_vtm_code']], how='left', on='vtm_code')
vmp_lkp = vmp_lkp[['vmp_code', 'reference_vmp_code', 'valid_date', 'previous_vmp_code',
       'vtm_code', 'reference_vtm_code','invalid_flag', 'vmp_desc', 'abbrev_desc',
       'preferred_desc_basis', 'desc_valid_date', 'previous_desc',
       'previous_desc_basis', 'desc_change_reason', 'combination_product',
       'prescribing_status', 'sugar_free', 'gluten_free', 'preservative_free',
       'cfc_free', 'amp_non_avail', 'amp_non_avail_date', 'dose_form',
       'unit_dose_form_size', 'unit_dose_form_units', 'unit_dose_meas_unit',
       'earliest_release', 'latest_release']]

Re-export everything

In [ ]:
vtm_lkp.to_csv(os.path.join(trans_folder,'vtm_lookup.csv'), index=False)
vmp_lkp.to_csv(os.path.join(trans_folder,'vmp_lookup.csv'), index=False)
amp_lkp.to_csv(os.path.join(trans_folder,'amp_lookup.csv'), index=False)
ampp_lkp.to_csv(os.path.join(trans_folder,'ampp_lookup.csv'), index=False)
vmpp_lkp.to_csv(os.path.join(trans_folder,'vmpp_lookup.csv'), index=False)

ampp_lkp.to_csv(os.path.join(lkp_folder,'dmd_ampp_info_lkp.csv.gz'), compression='gzip', index=False)
vmpp_lkp.to_csv(os.path.join(lkp_folder,'dmd_vmpp_info_lkp.csv.gz'), compression='gzip', index=False)
amp_lkp.to_csv(os.path.join(lkp_folder,'dmd_amp_info_lkp.csv.gz'), compression='gzip', index=False)
vmp_lkp.to_csv(os.path.join(lkp_folder,'dmd_vmp_info_lkp.csv.gz'), compression='gzip', index=False)
vtm_lkp.to_csv(os.path.join(lkp_folder,'dmd_vtm_lkp.csv.gz'), compression='gzip', index=False)

dmd_desc_lkp.to_csv(os.path.join(lkp_folder,'dmd_desc_type_lkp.csv.gz'), compression='gzip', index=False)

Simple singular AMP to VMP lkp

In [ ]:
# some AMP codes have 2 VMP codes but they are largely the same
# e.g., 'Lymphoedema garments' and 'Lymphoedema garments below knee open toe'
# so just picking the last one using default dict(zip()) functionality
dmd_amp_vmp_map = dict(zip(amp_lkp["amp_code"], amp_lkp["reference_vmp_code"]))
pd.DataFrame(dmd_amp_vmp_map.items(), columns=["amp_code", "reference_vmp_code"]).to_csv(os.path.join(lkp_folder,'dmd_amp_to_vmp_map.csv.gz'), compression='gzip', index=False)

Also make a super simple file (ignoring duplicate rows from descriptions) to make join/map easier

In [ ]:
dmd_desc_lkp = pd.read_csv(os.path.join(lkp_folder,'dmd_desc_type_lkp.csv.gz'), compression='gzip', dtype=object)

In [ ]:
simple_dmd_map = dmd_desc_lkp[['dmd_code','reference_code']].drop_duplicates(ignore_index=True)

In [ ]:
simple_dmd_map['dmd_code'].nunique() == len(simple_dmd_map)

In [ ]:
simple_dmd_map.to_csv(os.path.join(lkp_folder,'dmd_reference_code_map.csv.gz'), compression='gzip', index=False)

### Build reference code to 'best' description lookup

There should be a single description per reference code. Descriptions can have multiple codes.


#### VTM

In [ ]:
vtm_lkp.groupby('vtm_desc', as_index=False)['reference_vtm_code'].nunique().sort_values('reference_vtm_code', ascending=False)

#### AMP approach

The important thing is that the AMP codes can be linked to the correct VMP code, ingredient, etc. So I need to be sure the reference vmp code is the same for all cases where there are multiple codes with the same description

In [ ]:
amp_lkp.groupby('amp_desc', as_index=False)['reference_vmp_code'].nunique().sort_values('reference_vmp_code', ascending=False)

In [ ]:
# check that the reference codes have unique descriptions
vtm_test = vtm_lkp[['reference_vtm_code','vtm_desc']].drop_duplicates()

In [ ]:
vtm_test['reference_vtm_code'].nunique() == vtm_test['vtm_desc'].nunique()

In [ ]:
vtm_to_inspect = vtm_test[vtm_test.duplicated(subset='vtm_desc',keep=False)].sort_values('vtm_desc')['reference_vtm_code']

In [ ]:
vtm_lkp[
    (vtm_lkp['reference_vtm_code'].isin(vtm_to_inspect))
    #&(vtm_lkp['latest_release']=='2024_7.2.0')
    ].sort_values(['vtm_desc','reference_vtm_code'])

## ATC map/lookup

Updated with same version as dm+d consolidated file (2024 version 7.2.0)

In [ ]:
atc_folder = os.path.join('//XXXX/XXXX','XXXX','TRUD_maps','dmd supplementary (ATC)','nhsbsa_dmdbonus_7.2.0_20240715000001')

tree = ET.parse(os.path.join(atc_folder,'week292024-r2_3-BNF','f_bnf1_0110724.xml'))
root = tree.getroot()

data = []
for vmp in root.findall('.//VMP'):
    vpid = vmp.find('VPID').text
    bnf = vmp.find('BNF').text if vmp.find('BNF') is not None else None
    atc = vmp.find('ATC').text if vmp.find('ATC') is not None else None
    data.append({'vmp_code': vpid, 'bnf_code': bnf, 'atc_code': atc})

bnf_atc_df = pd.DataFrame(data)

In [ ]:
bnf_atc_df.head()

In [ ]:
bnf_atc_df.to_csv(os.path.join(ROOT_DIR, 'external_data','lookups','EHR','primary_care','dmd_vmp_bnf_atc_lkp.csv'), index=False)

#### ATC lookup

downloaded the file directly from https://github.com/fabkury/atcd/blob/master/WHO%20ATC-DDD%202024-07-31.csv into `external_data > lookups > EHR > primary_care > atc_lookup.csv` and `S: > TRUD_maps > dmd supplementary (ATC)`

## BNF lookup

Make simple version of lookup file

In [ ]:
# BNF lookup
bnf_path_2019 = os.path.join(lkp_folder,'20231128_1701153622884_BNF_Code_Information.csv.gz')
bnf_lkp = pd.read_csv(bnf_path_2019,dtype='object',compression='gzip')

bnf_levels = [x.replace('BNF ','') for x in bnf_lkp.columns if 'Code' not in x]

bnf_key = pd.DataFrame()
for i, level in enumerate(bnf_levels):
    # get unique codes per level
    temp_df = (bnf_lkp
               [[f'BNF {level} Code',f'BNF {level}']]
               .drop_duplicates()
               .rename(columns={f'BNF {level} Code':'bnf_code',f'BNF {level}':'desc'}))
    temp_df.insert(2,'bnf_type',f'{i+1}_{level.lower()}')
    bnf_key = pd.concat([bnf_key,temp_df], ignore_index=True)

# export
bnf_key.to_csv(os.path.join(lkp_folder,'bnf_key.csv.gz'),compression='gzip',index=False)

See coverage by getting list of unique BNF codes to map, as well as the number of entries represented

In [ ]:
sql = f"""
SELECT
    bnf_code
FROM
    gp_prescriptions
WHERE
    dmd_code IS NULL AND
    bnf_code IS NOT NULL;"""

bnf_to_map = pd.DataFrame(Database.query(sql).fetchall(), columns=['bnf_code'])
bnf_to_map.rename(columns={'bnf_code':'original_bnf_code'}, inplace=True)

In [ ]:
# standardise code format (to match SNOMED and BNF lookup)
def format_bnf(code):
    # drop trailing 0s
    formatted_code = code.replace('.00','').replace('.','')
    # if still has trailing zeroes
    if formatted_code[-2:]=='00':
        return formatted_code[:-2]
    else:
        return formatted_code
bnf_to_map['bnf_code'] = bnf_to_map['original_bnf_code'].apply(format_bnf)

# get unique combos to serve as key
bnf_format_key = bnf_to_map.drop_duplicates(ignore_index=True)
# export this, to be uploaded into SQL for easy merging later
bnf_format_key.to_csv(os.path.join(lkp_folder,'bnf_from_entries_formatted_key.csv'), index=False)

In [ ]:
# get counts per formatted code
bnf_codes_counts = bnf_to_map.groupby('bnf_code', as_index=False).size().rename(columns={'size':'n_entries'})
# drop dummy code
bnf_codes_counts = bnf_codes_counts[bnf_codes_counts.bnf_code!='000000'].reset_index(drop=True)

## Welsh Read info (and dm+d map) via UMLS

In [ ]:
welsh_codes = pd.read_csv('Welsh_codes_tomap.csv')

In [ ]:
# extract code info
welsh_lkp_path = os.path.abspath(os.path.join('P:','atlas','umls_data','welsh_prescription_lookup_fromUKBentries.csv'))
welsh_lkp_file = open(welsh_lkp_path,'w')
welsh_lkp_file.write('CUI,CODE,TTY,STR\n')

umls_path = os.path.abspath(os.path.join('X:','XXXX','umls_source_files','MRCONSO.RRF.ENG'))
count=0
for line in tqdm(open(umls_path, encoding='latin-1')):
    vals = line.split('|')
    code = vals[13]
    sab = vals[11] # read code = RCD
    if (code in welsh_codes)&(sab=='RCD'):
        cui = vals[0]
        term = vals[14].replace(',','')
        tty = vals[12]
        welsh_lkp_file.write(f'{cui},{code},{tty},{term}\n')
welsh_lkp_file.close()

In [ ]:
# re-read
welsh_lkp = pd.read_csv(os.path.abspath(os.path.join('X:','XXXX','umls_data','welsh_prescription_lookup_fromUKBentries.csv')))
# get SNOMED codes for the same concepts
welsh_cui = list(welsh_lkp['CUI'])

snomed_welsh_cui_path = os.path.abspath(os.path.join('X:','XXXX','umls_data','all_SNOMED_codes_w_same_Welsh_prescription_CUIs.csv'))
snomed_welsh_cui = open(snomed_welsh_cui_path,'w')
snomed_welsh_cui.write('CUI,SAB,CODE,TTY,STR\n')

umls_path = os.path.abspath(os.path.join('X:','XXXX','umls_source_files','MRCONSO.RRF.ENG'))
count=0
for line in tqdm(open(umls_path, encoding='latin-1')):
    vals = line.split('|')
    cui = vals[0]
    code = vals[13]
    sab = vals[11]
    if (cui in welsh_cui)&(sab == 'SNOMEDCT_US'):
        term = vals[14].replace(',','')
        tty = vals[12]
        snomed_welsh_cui.write(f'{cui},{sab},{code},{tty},{term}\n')
snomed_welsh_cui.close()

##### Choose the best term per code
PT = preferred term, AB = abbreviation, OP = Obsolete preferred, OA = Obsolete abbreviation, SY = synonym, IS = obsolete synonym

https://www.nlm.nih.gov/research/umls/knowledge_sources/metathesaurus/release/abbreviations.html#mrdoc_TTY

In [ ]:
# import files
welsh_lkp_path = os.path.abspath(os.path.join('X:','XXXX','umls_data','welsh_prescription_lookup_fromUKBentries.csv'))
welsh_lkp = pd.read_csv(welsh_lkp_path,dtype=object)
welsh_lkp.sort_values('CODE', ignore_index=True, inplace=True)
snomed_welsh_cui_path = os.path.abspath(os.path.join('X:','XXXX','umls_data','all_SNOMED_codes_w_same_Welsh_prescription_CUIs.csv'))
snomed_welsh_cui = pd.read_csv(snomed_welsh_cui_path, dtype=object)
snomed_welsh_cui = snomed_welsh_cui.drop(columns=['SAB'])

In [ ]:
# WELSH LOOKUP - first PT then OP
pt_welsh_lkp = welsh_lkp[welsh_lkp.TTY=='PT']
op_welsh_lkp = welsh_lkp[(welsh_lkp.TTY=='OP')&(~welsh_lkp.CODE.isin(pt_welsh_lkp['CODE']))]
pref_welsh_lkp = pd.concat([pt_welsh_lkp,op_welsh_lkp]).sort_values('CODE',ignore_index=True)
assert len(set(welsh_lkp['CODE'].unique()) - set(pref_welsh_lkp['CODE'].unique())) == 0
pref_welsh_lkp = pref_welsh_lkp.drop(columns=['TTY']).rename(columns={'CODE':'welsh_read_code','STR':'welsh_read_term'})

In [ ]:
# SNOMED LOOKUP - first PT
pt_snomed_welsh_cui = snomed_welsh_cui[snomed_welsh_cui.TTY=='PT']
# then OP
op_snomed_welsh_cui = snomed_welsh_cui[(snomed_welsh_cui.TTY=='OP')&(~snomed_welsh_cui.CUI.isin(pt_snomed_welsh_cui['CUI']))]
pref_snomed_welsh_cui = pd.concat([pt_snomed_welsh_cui,op_snomed_welsh_cui]).sort_values('CUI',ignore_index=True)
# then OAP
oap_snomed_welsh_cui = snomed_welsh_cui[(~snomed_welsh_cui.CUI.isin(pref_snomed_welsh_cui.CUI))&(snomed_welsh_cui.TTY=='OAP')]
pref_snomed_welsh_cui = pd.concat([pref_snomed_welsh_cui,oap_snomed_welsh_cui]).sort_values('CUI',ignore_index=True)
# then OAF
oaf_snomed_welsh_cui = snomed_welsh_cui[(~snomed_welsh_cui.CUI.isin(pref_snomed_welsh_cui.CUI))&(snomed_welsh_cui.TTY=='OAF')]
pref_snomed_welsh_cui = pd.concat([pref_snomed_welsh_cui,oaf_snomed_welsh_cui]).sort_values('CUI',ignore_index=True)
# then OF
of_snomed_welsh_cui = snomed_welsh_cui[(~snomed_welsh_cui.CUI.isin(pref_snomed_welsh_cui.CUI))&(snomed_welsh_cui.TTY=='OF')]
pref_snomed_welsh_cui = pd.concat([pref_snomed_welsh_cui,of_snomed_welsh_cui]).sort_values('CUI',ignore_index=True)
# then IS
is_snomed_welsh_cui = snomed_welsh_cui[(~snomed_welsh_cui.CUI.isin(pref_snomed_welsh_cui.CUI))&(snomed_welsh_cui.TTY=='IS')]
pref_snomed_welsh_cui = pd.concat([pref_snomed_welsh_cui,is_snomed_welsh_cui]).sort_values('CUI',ignore_index=True)
# then OAS
oas_snomed_welsh_cui = snomed_welsh_cui[(~snomed_welsh_cui.CUI.isin(pref_snomed_welsh_cui.CUI))&(snomed_welsh_cui.TTY=='OAS')]
pref_snomed_welsh_cui = pd.concat([pref_snomed_welsh_cui,oas_snomed_welsh_cui]).sort_values('CUI',ignore_index=True)

assert snomed_welsh_cui['CUI'].nunique() == pref_snomed_welsh_cui['CUI'].nunique()

# remove duplicate terms
pref_snomed_welsh_cui = pref_snomed_welsh_cui.drop_duplicates(subset=['CUI','CODE'])
pref_snomed_welsh_cui = pref_snomed_welsh_cui.drop(columns=['TTY']).rename(columns={'CODE':'snomed_code','STR':'snomed_term'})

In [ ]:
# combine
welsh_lkp_dmd = pref_welsh_lkp.merge(pref_snomed_welsh_cui, how='left')
# add number of entries in UKB
welsh_hits = pd.read_csv('Welsh_codes_tomap.csv')
welsh_lkp_dmd = welsh_hits.merge(welsh_lkp_dmd, how='left', right_on='welsh_read_code', left_on='read_2')
welsh_lkp_dmd = welsh_lkp_dmd.drop(columns=['welsh_read_code']).rename(columns={'read_2':'welsh_read_code','size':'n_ukb_entries'})

In [ ]:
# legibility
welsh_lkp_dmd.rename(columns={'welsh_read_term':'umls_read_desc','snomed_term':'umls_dmd_desc','snomed_code':'umls_dmd_code','CUI':'umls_cui'}, inplace=True)

In [ ]:
# export
welsh_lkp_dmd.to_csv(os.path.abspath(os.path.join('P:','atlas','umls_data','welsh_prescription_lookup_fromUKBentries_mapped_to_SNOMED.csv')),index=False)
welsh_lkp_dmd.to_csv(os.path.join(lkp_folder,'welsh_read_from_entries_lkp_dmd_map_via_UMLS.csv'), index=False)

## Welsh description lookup

From the UMLS mapping

In [ ]:
welsh_desc_from_lkp = (welsh_lkp
                       [~welsh_lkp.welsh_read_term.isna()]
                       [['welsh_read_code','welsh_read_term']]
                       .drop_duplicates()
                       .rename(columns={'welsh_read_code':'read_code','welsh_read_term':'desc'}))
welsh_desc_from_lkp['source'] = 'umls'

In [ ]:
# filter codes with descriptions in the entries for codes remaining to be described
read_to_get_desc = list(welsh_lkp[~welsh_lkp.welsh_read_code.isin(welsh_desc_from_lkp.read_code)]['welsh_read_code'].unique())
read_desc_from_entries = read_no_desc_entries[read_no_desc_entries.read_code.isin(read_to_get_desc)]
# arbitrarily use the longest description per code
read_desc_from_entries['len'] = read_desc_from_entries['drug_name'].apply(len)
read_desc_from_entries = (read_desc_from_entries
                          .sort_values('len',ascending=False)
                          .drop_duplicates(subset='read_code',keep='first')
                          .drop(columns='len')
                          .rename(columns={'drug_name':'desc'}))
read_desc_from_entries['source'] = 'entries'
# combine
welsh_desc_lkp = pd.concat([welsh_desc_from_lkp,read_desc_from_entries])

In [ ]:
# add back those without a desc by this procedure
read_wo_desc = (welsh_lkp
                [~welsh_lkp.welsh_read_code.isin(welsh_desc_lkp.read_code)]
                [['welsh_read_code']]
                .rename(columns={'welsh_read_code':'read_code'}))
read_wo_desc['desc'] = np.nan
read_wo_desc['source'] = np.nan
# combine
welsh_desc_lkp = pd.concat([welsh_desc_lkp,read_wo_desc]).sort_values('read_code',ignore_index=True)
# export
welsh_desc_lkp.to_csv(os.path.join(lkp_folder,'welsh_read_desc_lkp.csv.gz'),index=False,compression='gzip')